In [ ]:
## ảnh bị lệch khi gắn vào source .md
import re
import shutil

MD_FILE = "giao-ly-va-truong-phai.md"
BACKUP  = "giao-ly-va-truong-phai.md.bak"
START_LINE = 1616          # dòng bắt đầu bị lệch (1-indexed)
OFFSET = 1                # lệch +1

# Backup trước khi sửa
shutil.copy(MD_FILE, BACKUP)
print(f"Đã backup: {BACKUP}")

# Pattern: ![img-N.ext1](images/img-M.ext2)
pattern = re.compile(
    r'!\[img-(\d+)(\.[a-zA-Z0-9]+)\]\(images/img-(\d+)(\.[a-zA-Z0-9]+)\)'
)

with open(MD_FILE, "r", encoding="utf-8") as f:
    lines = f.readlines()

fixed = 0

def repl(m):
    global fixed
    n_alt  = int(m.group(1))
    ext1   = m.group(2)
    n_path = int(m.group(3))
    ext2   = m.group(4)
    fixed += 1
    return f'![img-{n_alt + OFFSET}{ext1}](images/img-{n_path + OFFSET}{ext2})'

# Chỉ sửa từ START_LINE trở đi
for i in range(START_LINE - 1, len(lines)):
    lines[i] = pattern.sub(repl, lines[i])

with open(MD_FILE, "w", encoding="utf-8") as f:
    f.writelines(lines)

print(f"Đã sửa {fixed} ảnh từ line {START_LINE} trở đi (offset +{OFFSET}).")

Đã backup: giao-ly-va-truong-phai.md.bak
Đã sửa 13 ảnh từ line 1616 trở đi (offset +1).


In [4]:
# slit file .md
import os
import re
import unicodedata

# ===== Cấu hình =====
MD_FILE = "giao-ly-va-truong-phai.md"
OUTPUT_DIR = "chunks"
# ====================

os.makedirs(OUTPUT_DIR, exist_ok=True)


def slugify(text: str) -> str:
    """Chuyển tiêu đề thành slug: bỏ dấu, thay space/ký tự đặc biệt bằng '-'."""
    # Bỏ dấu tiếng Việt
    text = unicodedata.normalize("NFD", text)
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    # Lowercase
    text = text.lower()
    # Thay ký tự không phải chữ/số bằng '-'
    text = re.sub(r"[^a-z0-9]+", "-", text)
    # Bỏ '-' ở đầu/cuối
    text = text.strip("-")
    # Giới hạn độ dài
    return text[:80] or "untitled"


# Đọc toàn bộ file
with open(MD_FILE, "r", encoding="utf-8") as f:
    lines = f.readlines()

# Tìm các dòng là H1: chỉ match '# ' ở đầu (không match '##')
h1_pattern = re.compile(r"^#\s+(.+?)\s*$")

sections = []          # list of (title, [lines])
current_title = None
current_lines = []

for line in lines:
    m = h1_pattern.match(line)
    if m:
        # Lưu section trước đó (nếu có)
        if current_title is not None:
            sections.append((current_title, current_lines))
        current_title = m.group(1).strip()
        current_lines = [line]   # giữ luôn dòng header H1
    else:
        if current_title is not None:
            current_lines.append(line)
        # Nếu chưa gặp H1 nào → bỏ qua (nội dung trước H1 đầu tiên)

# Lưu section cuối
if current_title is not None:
    sections.append((current_title, current_lines))

print(f"Tìm thấy {len(sections)} section (H1).")

# Ghi từng file
for idx, (title, sec_lines) in enumerate(sections, start=1):
    slug = slugify(title)
    filename = f"{idx:02d}-{slug}.md"
    filepath = os.path.join(OUTPUT_DIR, filename)

    # Chuẩn hoá: bỏ dòng trống thừa ở đầu/cuối
    content = "".join(sec_lines).strip() + "\n"

    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

    print(f"[{idx:02d}] {filename}  ({len(sec_lines)} dòng)")

print(f"\nĐã ghi {len(sections)} file vào folder: {OUTPUT_DIR}/")

Tìm thấy 11 section (H1).
[01] 01-preface.md  (21 dòng)
[02] 02-introduction.md  (23 dòng)
[03] 03-i-the-buddha-s-life.md  (174 dòng)
[04] 04-ii-hinayana-the-buddhism-of-liberation-through-self-effort.md  (851 dòng)
[05] 05-iii-mahayana-the-monistic-buddhism-of-liberation-by-other-power-19.md  (710 dòng)
[06] 06-iv-philosophical-schools-of-the-mahayana.md  (193 dòng)
[07] 07-v-the-tantrayana-and-the-buddhism-of-east-asia.md  (108 dòng)
[08] 08-vi-a-cultural-historical-survey.md  (119 dòng)
[09] 09-vii-tabulated-synopsis.md  (46 dòng)
[10] 10-viii-literature.md  (166 dòng)
[11] 11-index.md  (1724 dòng)

Đã ghi 11 file vào folder: chunks/
